# 🐭 マウスひっかき行動自動検出システム — Google Colab 版

本ノートブックは、**ボタンを押すだけ**でマウスのひっかき行動を自動検出できるシステムを Google Colab 上で動作させます。

### 動作フロー
```
実験動画 (.mp4/.mov) または SLEAP .h5 ファイル
       ↓
   姿勢推定 (DeepLabCut SuperAnimal / YOLOv8-pose / SLEAP)
       ↓
   カラーID による個体識別（尾の色＋帯本数）
       ↓
   物理単位 (mm/s, mm) のルールベース判定でひっかき行動を検出 & Bout 集計
       ↓
   Web ダッシュボードで可視化・手動補正・CSV エクスポート
```

### 出力されるファイル（`outputs/<動画名>_*`）
| ファイル | 内容 |
|----------|------|
| `*_frame_labels.csv` | フレームごとの姿勢・運動量・ひっかき確率 |
| `*_bouts.csv` | 検出された各 bout（ひっかき動作のまとまり）の開始/終了/持続時間 |
| `*_summary.csv` | 個体ごとの集計指標（bout 数・総スクラッチ秒・頻度・最長 など） |
| `*_columns.csv` | **データ辞書**（各列の単位・意味・算出式を日本語で記載。Excel で開ける UTF-8 BOM 付き） |
| `*_id_quality.json` | 個体ID識別の品質情報（swap イベント等） |
| `*_provenance.json` | 再現性のための実行メタデータ（エンジン・モデル・閾値・px→mm 係数 など） |

### 推奨実行手順
1. ランタイム → ランタイムのタイプを変更 → **GPU (T4)** を選択
2. 各セルを上から順に実行 (`Shift+Enter`)
3. ステップ7 で発行される URL をクリックしてダッシュボードを開く
4. 高精度なマウス専用推論を使うなら **ステップ11 (DeepLabCut SuperAnimal)** を実行

> **ローカル Windows 版との違い**: ローカル版（`start.bat` / `check_env.py`）は RTX 5080（sm_120）対応のため PyTorch を **cu128** ホイールで導入しますが、Colab の GPU（T4 = sm_75 など）は標準で配布されている PyTorch がそのまま動くため、**cu128 への入れ直しは不要**です。本ノートブックは Colab の既存 PyTorch を尊重します。

---


## ⚡ ステップ0: GPU 確認
T4 GPU が割り当てられているか確認します。GPU なしでも動作しますが、姿勢推定が遅くなります。


In [ ]:
import subprocess, sys, os

print('=' * 55)
print(' GPU / 環境チェック')
print('=' * 55)

# ── GPU 確認 ─────────────────────────────────────────────
GPU_AVAILABLE = False
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                             '--format=csv,noheader'], capture_output=True, text=True)
    if result.returncode == 0:
        GPU_AVAILABLE = True
        print(f'✅ GPU 検出: {result.stdout.strip()}')
    else:
        print('⚠️  GPU が検出されませんでした。CPU モードで動作します（姿勢推定が遅くなります）。')
        print('   [ランタイム] → [ランタイムのタイプを変更] → GPU を選択すると高速化できます。')
except FileNotFoundError:
    print('⚠️  nvidia-smi が見つかりません。CPU モードで動作します。')

# ── Python / Colab 環境確認 ──────────────────────────────
print(f'✅ Python: {sys.version.split()[0]}')
try:
    import google.colab  # noqa: F401
    print('✅ Google Colab 環境を検出しました')
    IN_COLAB = True
except ImportError:
    print('ℹ️  ローカル Jupyter 環境として動作します')
    IN_COLAB = False

# ── PyTorch / CUDA 確認（Colab には標準で導入済み）────────
#   ローカル版 check_env.py と異なり、Colab の GPU (T4=sm_75 等) は
#   標準配布の PyTorch でカーネルが揃っているため cu128 への入れ直しは不要。
#   ここでは「torch から実際に CUDA が見えているか」だけを確認する。
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    if cuda_ok:
        cap = torch.cuda.get_device_capability(0)
        print(f'✅ PyTorch {torch.__version__} / CUDA 利用可: True '
              f'/ デバイス: {torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]})')
    else:
        print(f'ℹ️  PyTorch {torch.__version__} は導入済みですが CUDA は無効です（CPU 実行）。')
except ImportError:
    print('ℹ️  PyTorch は未導入です（ステップ3/11 で必要に応じて導入されます）。')

print('=' * 55)

# グローバル変数として保存
LOCAL_PROJECT_PATH = '/content/mouse-scratch-dashboard'
print(f'プロジェクトパス: {LOCAL_PROJECT_PATH}')


---
## 🛠️ ステップ1: Google Drive の接続
実験動画や SLEAP データを Google Drive から直接読み込むため、Drive をマウントします。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive をマウントしました')


---
## 📂 ステップ2: プロジェクトフォルダの配置
Google Drive 上のプロジェクトフォルダを、Colab のローカル高速ディスク (`/content`) にコピーします。

> **`DRIVE_PROJECT_PATH`** を自分の Google Drive 上のパスに変更してください。


In [ ]:
import os, shutil

# ▼ ご自身の Google Drive 上のパスを指定してください
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/mouse-scratch-dashboard'
LOCAL_PROJECT_PATH = '/content/mouse-scratch-dashboard'

if os.path.exists(DRIVE_PROJECT_PATH):
    if os.path.exists(LOCAL_PROJECT_PATH):
        shutil.rmtree(LOCAL_PROJECT_PATH)
    shutil.copytree(DRIVE_PROJECT_PATH, LOCAL_PROJECT_PATH)
    os.chdir(LOCAL_PROJECT_PATH)
    print(f'✅ プロジェクトをコピーしました: {LOCAL_PROJECT_PATH}')
    print(f'   作業ディレクトリ: {os.getcwd()}')
else:
    print(f'❌ フォルダが見つかりません: {DRIVE_PROJECT_PATH}')
    print('   Google Drive に mouse-scratch-dashboard フォルダをアップロードしてください。')
    raise FileNotFoundError(DRIVE_PROJECT_PATH)


---
## 📦 ステップ3: 依存ライブラリのインストール
FastAPI サーバー・YOLOv8・トンネリングツールをインストールします。
パッケージのバージョンは、ローカル版 `check_env.py` の `BASE_PACKAGES` と揃えてあります（`numpy<2` は DeepLabCut の要件）。

> **姿勢推定エンジンの選び方**
> - **DeepLabCut SuperAnimal**（推奨・マウス専用・追加学習不要）→ ステップ3 完了後に **ステップ11** を実行
> - **YOLOv8**（軽量・高速・このステップで自動導入）
> - **SLEAP**（TF 依存の競合が起きやすいため既定では導入しません。必要時のみステップ8 で個別導入）


In [ ]:
import subprocess, sys, importlib.util

def pip_install(*packages, quiet=True):
    cmd = [sys.executable, '-m', 'pip', 'install'] + list(packages)
    if quiet:
        cmd += ['-q', '--quiet']
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0, result.stderr

def _has(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

# ── 基本パッケージ（ローカル版 check_env.py の BASE_PACKAGES と一致させること）──
#   numpy<2 は DeepLabCut の要件。pandas/h5py は CSV/H5 入出力に必須。
print('📦 [1/3] コアライブラリをインストール中...')
ok, err = pip_install('fastapi>=0.111.0', 'uvicorn[standard]>=0.29.0', 'pydantic>=2.7.0',
                      'opencv-python-headless>=4.9.0', 'numpy>=1.23.0,<2.0',
                      'pandas>=2.0.0', 'h5py>=3.9.0', 'httpx>=0.27.0')
print('  ✅ コアライブラリ' if ok else f'  ⚠️ エラー: {err[-300:]}')

print('📦 [2/3] YOLOv8 (ultralytics) をインストール中...')
ok, err = pip_install('ultralytics>=8.2.0')
print('  ✅ ultralytics (YOLOv8)' if ok else f'  ⚠️ エラー: {err[-300:]}')

print('📦 [3/3] localtunnel (公開 URL 用) をインストール中...')
result = subprocess.run(['npm', 'install', '-g', 'localtunnel'],
                        capture_output=True, text=True)
print('  ✅ localtunnel' if result.returncode == 0 else f'  ⚠️ npm エラー: {result.stderr[-200:]}')

# ── 最終確認（check_env.py の verify() と同等の表示）────────
print('\n── 依存チェック ──')
for mod in ['fastapi', 'uvicorn', 'pydantic', 'cv2', 'numpy', 'pandas', 'h5py', 'httpx', 'ultralytics', 'torch']:
    print(f'  {"OK " if _has(mod) else "NG "} {mod}')

print('\nℹ️  SLEAP は TF 依存の競合が起きやすいため既定では導入しません。')
print('   必要な場合のみ「ステップ8」で個別に導入してください。')
print('ℹ️  DeepLabCut SuperAnimal（推奨のマウス専用モデル）は「ステップ11」で導入します。')
print('\n✅ インストール完了')


---
## 🤖 ステップ4: モデルの準備確認
YOLOv8 デフォルトモデル (`yolov8n-pose.pt`) の準備状況を確認します。
初回は自動ダウンロードされます（約 6MB）。


In [ ]:
import os, sys
os.chdir(LOCAL_PROJECT_PATH)

# setup_models.py を実行してモデルの準備状況を確認・表示
result = subprocess.run([sys.executable, 'setup_models.py'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('[stderr]', result.stderr[-500:])


---
## 🌐 ステップ5: Colab インスタンスの IP アドレスを確認
localtunnel の認証に使用します。表示された IP アドレスを手元に控えてください。


In [ ]:
print('=' * 55)
print(' localtunnel 認証用 IP アドレス')
print('=' * 55)
!curl -s ipv4.icanhazip.com
print('\n↑ このIPアドレスをステップ7でコピー＆ペーストします')
print('=' * 55)


---
## 🚀 ステップ6: FastAPI バックエンドサーバーの起動
ダッシュボードのバックエンドをバックグラウンドで起動します。


In [ ]:
import subprocess, time, os, sys

os.chdir(LOCAL_PROJECT_PATH)

print('FastAPI サーバーを起動しています...')
server_process = subprocess.Popen(
    [sys.executable, 'app_server.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    errors='replace'
)

time.sleep(4)

# 起動ログを表示
print('--- サーバー起動ログ ---')
for _ in range(8):
    line = server_process.stdout.readline()
    if line:
        print(f'  {line.rstrip()}')
print('------------------------')

# 動作確認
import urllib.request
try:
    with urllib.request.urlopen('http://localhost:8000/', timeout=5) as r:
        print(f'\n✅ サーバー起動確認: HTTP {r.status}')
except Exception as e:
    print(f'\n⚠️  接続確認に失敗しました: {e}')
    print('   少し待ってから次のステップに進んでください。')


---
## 🔗 ステップ7: 外部公開 URL の生成
**方法 A**: localtunnel（推奨。IPアドレスの入力が必要）

**方法 B**: Cloudflare Tunnel（より安定。IPアドレス入力不要）

どちらか片方を実行してください。


In [ ]:
# ── 方法 A: localtunnel ──────────────────────────────────────────
import subprocess, time

print('🔗 [方法 A] localtunnel でトンネルを開いています...')
lt_process = subprocess.Popen(
    ['lt', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(6)

url = None
for _ in range(10):
    line = lt_process.stdout.readline()
    if 'your url is' in line.lower():
        url = line.split('is:')[-1].strip()
        break

print('\n' + '=' * 65)
print('🌐 ダッシュボードへのアクセス手順 (localtunnel)')
print('=' * 65)
if url:
    print(f'\n  👉  {url}  👈\n')
else:
    print('  ⚠️  URL の取得に失敗しました。少し待ってから再実行してください。')
    print('  または「方法 B」(Cloudflare) セルを実行してください。')
print('手順:')
print('  1. 上のURLをクリック')
print('  2. 表示される入力欄にステップ5の IP アドレスを入力')
print('  3. "Click to Submit" をクリック → ダッシュボードが表示されます')
print('=' * 65)


In [ ]:
# ── 方法 B: Cloudflare Tunnel (IP 入力不要・より安定) ──────────
import subprocess, threading, time, re, os

# cloudflared のダウンロード (まだなければ)
cf_bin = '/usr/local/bin/cloudflared'
if not os.path.exists(cf_bin):
    print('📥 cloudflared をダウンロード中...')
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', cf_bin
    ], check=True)
    subprocess.run(['chmod', '+x', cf_bin], check=True)
    print('✅ cloudflared を準備しました')

url_found = []

def _read_cf(proc):
    pattern = re.compile(r'https://[a-z0-9\-]+\.trycloudflare\.com')
    for line in proc.stderr:
        m = pattern.search(line)
        if m:
            url_found.append(m.group())
            break

cf_proc = subprocess.Popen(
    [cf_bin, 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True
)

t = threading.Thread(target=_read_cf, args=(cf_proc,), daemon=True)
t.start()
t.join(timeout=20)

print('\n' + '=' * 65)
print('🌐 ダッシュボードへのアクセス (Cloudflare Tunnel)')
print('=' * 65)
if url_found:
    print(f'\n  👉  {url_found[0]}  👈\n')
    print('  このURLをクリックするだけでダッシュボードが開きます。')
    print('  (IP アドレスの入力は不要です)')
else:
    print('  ⚠️  URL の取得に失敗しました。「方法 A」を試してください。')
print('=' * 65)


---
## 💡 ダッシュボードの使い方

### 動画を解析する
ダッシュボード左パネルの「解析の実行 (Run Analysis)」に動画パスを入力して実行します。

**Google Drive 上の動画の場合:**
```
/content/drive/MyDrive/mouse_videos/cage_01.mp4
```
(Colab 左パネルのファイルブラウザで右クリック →「パスをコピー」で取得できます)

**SLEAP .h5 ファイルを既にお持ちの場合:**
- エンジンに `sleap` を選択し、動画パスと H5 ファイルパスを指定してください
- またはノートブック下部の「ステップ9」で直接 pipeline.py を呼び出すこともできます

### 手動補正
- Canvas 上の関節点をドラッグして位置を修正
- 「Bulk Offset」でフレーム全体のオフセットを一括適用
- 「Swap ID」で個体の取り違えを即座に修正
- 右上「Export CSV」でデータをダウンロード

---


---
## 📊 出力指標（summary）の意味

`outputs/<動画名>_summary.csv` には個体ごとの集計指標が出力されます。各列の定義は同じフォルダに自動生成される
**`<動画名>_columns.csv`（データ辞書）** にも日本語で記載されます（Excel で開ける UTF-8 BOM 付き）。主な指標は以下のとおりです。

| 指標 | 単位 | 意味 | 算出式 |
|------|------|------|--------|
| **bout 数** (`n_bouts`) | 回 | スクラッチ bout（連続したひっかき動作のまとまり）の検出総数 | その個体の bout 行数 |
| **総スクラッチ** (`total_scratch_duration_s`) | 秒 | 全 bout の合計時間。観察中に実際にひっかいていた総時間 | Σ bout_duration_s |
| **bout 率** (`scratch_rate_bouts_per_min`) | 回/分 | ひっかき頻度。1 分あたり何回の bout が起きたか（観察時間で正規化） | n_bouts / observation_duration_min |
| **強度** (`scratch_rate_s_per_min`) | 秒/分 | 1 分あたり何秒ひっかいていたか（観察時間で正規化） | total_scratch_duration_s / observation_duration_min |
| **最長** (`longest_bout_s`) | 秒 | 最も長く続いた単一 bout の持続時間 | max(bout_duration_s) |
| 平均 (`mean_bout_duration_s`) | 秒 | 1 回の bout の平均持続時間 | mean(bout_duration_s) |
| 中央値 (`median_bout_duration_s`) | 秒 | bout 持続時間の中央値（外れ値に頑健） | median(bout_duration_s) |
| 標準誤差 (`sem_bout_duration_s`) | 秒 | bout 持続時間の標準誤差（個体内ばらつき） | std(bout_duration_s) / √n_bouts |

> **観察時間で正規化する理由**: 動画の長さが個体・実験ごとに異なるため、`bout 率`（回/分）と `強度`（秒/分）を用いると条件をまたいで公平に比較できます。生の `bout 数` や `総スクラッチ秒` は観察時間が同じ場合のみ直接比較してください。

---


---
## 🔬 ステップ8 (応用): 複数モデルによる一括比較解析

同一動画に対して以下の手法で姿勢推定を実行し、ダッシュボードで精度比較を行います。

| 手法 | エンジン | モデル | キーポイント数 | 特徴 |
|------|----------|--------|---------------|------|
| 1 | DLC SuperAnimal | superanimal_topviewmouse (zero-shot) | 27 | 真上視点・マウス専用・追加学習不要 |
| 2 | DLC SuperAnimal | superanimal_quadruped (zero-shot) | 39 | 側面視点・**paws を含む**・最も解剖学的に正確 |
| 3 | SLEAP | Zenodo SuperAnimal-TopViewMouse-5K | 27 | 学術用大規模事前学習 |
| 4 | SLEAP | SLEAP 公式チュートリアル | テンプレート | 標準テンプレート |
| 5 | YOLOv8 | yolov8n-pose.pt (人間用ベース) | 17 (流用) | 軽量・高速・解剖学的精度は低い |
| 6 | YOLOv8 | Roboflow マウスデータで追加学習 | カスタム | 高速かつカスタム精度 |

> **DLC SuperAnimal は追加学習が不要で、本リポジトリで最も簡単かつ精度の高いマウス推定手段です (ステップ11 で詳説)。**


In [ ]:
# 比較検証用の追加パッケージ
print('📦 追加パッケージのインストール中...')
!pip install -q ultralytics roboflow h5py

# SLEAP (まだインストールされていない場合)
try:
    import sleap
    print(f'✅ SLEAP v{sleap.__version__} は既にインストール済みです')
except ImportError:
    print('📦 SLEAP をインストール中 (数分かかります)...')
    !pip install -q sleap
    try:
        import sleap
        print(f'✅ SLEAP v{sleap.__version__} インストール完了')
    except ImportError:
        print('⚠️ SLEAP のインストールに失敗しました。手法1・2はスキップされます。')


In [ ]:
# ── 手法1: Zenodo SuperAnimal-TopViewMouse-5K モデルのダウンロード ──
import os, urllib.request

os.chdir(LOCAL_PROJECT_PATH)
zenodo_url = 'https://zenodo.org/records/13757509/files/TopViewMouse5K_20240913.tar.gz?download=1'
tar_path   = '/content/TopViewMouse5K.tar.gz'
model_dir  = f'{LOCAL_PROJECT_PATH}/models/zenodo_mouse_5k'

os.makedirs(model_dir, exist_ok=True)

if not os.path.exists(tar_path):
    print('📥 Zenodo モデル (1.4 GB) をダウンロード中...')
    urllib.request.urlretrieve(zenodo_url, tar_path)
    print('✅ ダウンロード完了。展開中...')
    !tar -xzf {tar_path} -C {model_dir}
    print(f'✅ 手法1 モデルを配置しました: {model_dir}')
else:
    print(f'✅ 手法1 モデルは既にダウンロード済みです')


In [ ]:
# ── 手法2: SLEAP 公式テンプレートモデルのダウンロード ──
import os, urllib.request

tutorial_url  = 'https://github.com/talmolab/sleap/raw/main/tests/data/labels/labels.slp'
tutorial_dest = f'{LOCAL_PROJECT_PATH}/models/sleap_tutorial.slp'

if not os.path.exists(tutorial_dest):
    urllib.request.urlretrieve(tutorial_url, tutorial_dest)
    print(f'✅ 手法2 SLEAP テンプレートを配置しました: {tutorial_dest}')
else:
    print(f'✅ 手法2 モデルは既に配置済みです')


In [ ]:
# ── 手法4: Roboflow データセットで YOLOv8-pose を追加学習 ──
# ▼ Roboflow アカウントの API キーを入力してください (無料)
#   https://roboflow.com/ でサインアップ → 右上のアイコン → API Keys
ROBOFLOW_API_KEY = 'YOUR_ROBOFLOW_API_KEY'

if ROBOFLOW_API_KEY == 'YOUR_ROBOFLOW_API_KEY':
    print('⚠️  ROBOFLOW_API_KEY が未設定のため、手法4 (ファインチューニング) をスキップします。')
    print('   手法3 (YOLOv8 標準モデル) は引き続き使用可能です。')
else:
    from roboflow import Roboflow
    from ultralytics import YOLO
    import shutil

    rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace('sl-h33qa').project('mouse_poseestimate')
    dataset = project.version(1).download('yolov8')
    print('✅ Roboflow データセットをダウンロードしました')

    print('🚀 YOLOv8-pose ファインチューニング (10 epochs) 開始...')
    model = YOLO('yolov8n-pose.pt')
    model.train(data=os.path.join(dataset.location, 'data.yaml'),
                epochs=10, imgsz=640, device=0,
                project='mouse_pose_yolo', name='finetuned')

    best_pt = 'mouse_pose_yolo/finetuned/weights/best.pt'
    dest_pt = f'{LOCAL_PROJECT_PATH}/models/yolov8_mouse_finetuned.pt'
    if os.path.exists(best_pt):
        shutil.copy2(best_pt, dest_pt)
        print(f'✅ 学習済みモデルを配置しました: {dest_pt}')


In [ ]:
# ── 同一動画に 4 モデルを一括適用 ──────────────────────────────
import os, sys

os.chdir(LOCAL_PROJECT_PATH)

# ▼ 解析対象の動画パスを指定してください
TARGET_VIDEO = f'{LOCAL_PROJECT_PATH}/1000.mov'

if not os.path.exists(TARGET_VIDEO):
    print(f'⚠️  動画ファイルが見つかりません: {TARGET_VIDEO}')
    print('   TARGET_VIDEO のパスを修正してから実行してください。')
else:
    zenodo_model   = f'{LOCAL_PROJECT_PATH}/models/zenodo_mouse_5k'
    tutorial_model = f'{LOCAL_PROJECT_PATH}/models/sleap_tutorial.slp'
    custom_model   = f'{LOCAL_PROJECT_PATH}/models/yolov8_mouse_finetuned.pt'

    tasks = [
        ('手法1: Zenodo SLEAP',  ['--engine', 'sleap', '--model_path', zenodo_model],
         'outputs/result_zenodo', os.path.exists(zenodo_model)),
        ('手法2: SLEAP チュートリアル', ['--engine', 'sleap', '--model_path', tutorial_model],
         'outputs/result_sleap_tutorial', os.path.exists(tutorial_model)),
        ('手法3: YOLOv8 標準',   ['--engine', 'yolov8', '--model_path', 'yolov8n-pose.pt'],
         'outputs/result_yolov8_standard', True),
        ('手法4: YOLOv8 カスタム', ['--engine', 'yolov8', '--model_path', custom_model],
         'outputs/result_yolov8_finetuned', os.path.exists(custom_model)),
    ]

    for i, (name, engine_args, out_dir, available) in enumerate(tasks, 1):
        if not available:
            print(f'  [{i}/4] {name}: モデルが見つからないためスキップ')
            continue
        print(f'\n🔥 [{i}/4] {name} で解析中...')
        cmd = [sys.executable, 'pipeline.py', '--video', TARGET_VIDEO,
               '--out_dir', out_dir] + engine_args
        result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
        if result.returncode == 0:
            print(f'  ✅ 完了: {out_dir}')
        else:
            print(f'  ⚠️ エラー: {result.stdout[-500:]}')

    print('\n✨ 全モデルの一括解析が完了しました！')
    print('ダッシュボードの「比較解析結果の自動ロード」ドロップダウンから結果を切り替えて比較できます。')


In [ ]:
import os, shutil

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/mouse-scratch-dashboard/outputs'
local_outputs    = os.path.join(LOCAL_PROJECT_PATH, 'outputs')

# CSV（データ・データ辞書）に加え、再現性のための JSON（provenance / id_quality）も保存する
SAVE_EXTS = ('.csv', '.json')

if not os.path.exists(local_outputs):
    print('❌ outputs/ フォルダが見つかりません。先に解析を実行してください。')
else:
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    copied = 0
    for root, dirs, files in os.walk(local_outputs):
        for f in files:
            if f.endswith(SAVE_EXTS):
                src = os.path.join(root, f)
                rel = os.path.relpath(src, local_outputs)
                dst = os.path.join(DRIVE_BACKUP_DIR, rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copy2(src, dst)
                tag = ' (データ辞書)' if f.endswith('_columns.csv') else ''
                print(f'  💾 {rel}{tag}')
                copied += 1
    if copied == 0:
        print('保存対象のファイル (.csv / .json) が見つかりません。')
    else:
        print(f'\n✅ {copied} 件のファイルを Google Drive に保存しました。')
        print(f'   保存先: {DRIVE_BACKUP_DIR}')
        print('   ※ *_columns.csv は各列の単位・意味・算出式を記したデータ辞書です。')


In [ ]:
import os, shutil

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/mouse-scratch-dashboard/outputs'
local_outputs    = os.path.join(LOCAL_PROJECT_PATH, 'outputs')

if not os.path.exists(local_outputs):
    print('❌ outputs/ フォルダが見つかりません。先に解析を実行してください。')
else:
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    copied = 0
    for root, dirs, files in os.walk(local_outputs):
        for f in files:
            if f.endswith('.csv'):
                src = os.path.join(root, f)
                rel = os.path.relpath(src, local_outputs)
                dst = os.path.join(DRIVE_BACKUP_DIR, rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copy2(src, dst)
                print(f'  💾 {rel}')
                copied += 1
    if copied == 0:
        print('保存対象の CSV ファイルが見つかりません。')
    else:
        print(f'\n✅ {copied} 件の CSV ファイルを Google Drive に保存しました。')
        print(f'   保存先: {DRIVE_BACKUP_DIR}')


---
## 🔧 ステップ10 (応用): SLEAP .h5 ファイルを直接インポートして解析

すでに SLEAP で姿勢推定済みの `.h5` ファイルがある場合、そのデータを直接使って
特徴量抽出とひっかき行動検出だけを実行できます。


In [ ]:
import os, sys
os.chdir(LOCAL_PROJECT_PATH)

# ▼ 以下を自分の動画・H5 ファイルのパスに変更してください
VIDEO_PATH = '/content/drive/MyDrive/mouse_videos/cage_01.mp4'
SLEAP_H5   = '/content/drive/MyDrive/mouse_videos/cage_01.analysis.h5'
OUT_DIR    = 'outputs/from_sleap_h5'

if not os.path.exists(SLEAP_H5):
    print(f'⚠️  H5 ファイルが見つかりません: {SLEAP_H5}')
    print('   パスを修正してから実行してください。')
else:
    cmd = [
        sys.executable, 'pipeline.py',
        '--video', VIDEO_PATH,
        '--sleap', SLEAP_H5,
        '--out_dir', OUT_DIR
    ]
    print(f'▶ 実行: {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
    print(result.stdout)
    if result.returncode != 0:
        print('[Error]', result.stderr[-1000:])


---
## 🐭 ステップ11 (推奨): DeepLabCut SuperAnimal による高精度マウス専用推論

**DeepLabCut Model Zoo** に公開されている **SuperAnimal** モデルは、研究グループ DLC Team によって学習された
マウス姿勢推定モデルで、**追加学習なし (zero-shot) でそのまま使えます**。

| モデル | キーポイント数 | 視点 | 含まれる解剖学的特徴 | 推奨用途 |
|--------|------------|------|----------------------|----------|
| `superanimal_topviewmouse` | 27 点 | **真上** | nose, ears, shoulders, hips, tail, mid-back | ケージを上から撮影した動画 |
| `superanimal_quadruped`    | 39 点 | **側面/斜め** | nose, ears, **paws (4 足全て)**, joints, tail | paws の解剖学的正確さが重要な解析 |

> **真上視点動画には TopView**, **paws が見える側面/斜め視点動画には Quadruped** を選択してください。
> TopView モデルは paw キーポイントを含まないため、`left_hind_paw` 等は `left_hip` の近似値となります。


In [ ]:
# DLC SuperAnimal-TopViewMouse で解析する (真上視点の動画用)
import os, sys, subprocess

os.chdir(LOCAL_PROJECT_PATH)

# 実行デバイスは GPU/CPU を自動判別（デバイス選択セル未実行でも安全に動作）
if 'device_args' in globals():
    _device = device_args()
else:
    try:
        import torch
        _device = ['--device', 'cuda' if torch.cuda.is_available() else 'cpu']
    except Exception:
        _device = ['--device', 'cpu']

# ▼ 真上視点で撮影された動画のパスを指定してください
#   Colab 左パネルのファイルブラウザで右クリック →「パスをコピー」で取得できます
TARGET_VIDEO_TOPVIEW = '/content/drive/MyDrive/mouse_videos/cage_01.mp4'

if not os.path.exists(TARGET_VIDEO_TOPVIEW):
    print(f'⚠️  動画ファイルが見つかりません: {TARGET_VIDEO_TOPVIEW}')
    print('   TARGET_VIDEO_TOPVIEW を自分の動画パスに変更してから実行してください。')
    print('   例: /content/drive/MyDrive/mouse_videos/cage_topview.mp4')
else:
    cmd = [
        sys.executable, 'pipeline.py',
        '--video', TARGET_VIDEO_TOPVIEW,
        '--engine', 'dlc',
        '--dlc-superanimal', 'superanimal_topviewmouse',
        '--out_dir', 'outputs/result_dlc_topview',
        # ▼ ご自身のカメラで実測した値に置き換えてください
        '--px-to-mm', '0.25',
    ] + _device
    print('▶ 実行:', ' '.join(cmd))
    print('   初回は SuperAnimal モデル (~数百MB) のダウンロードに時間がかかります。')
    print('   ログがリアルタイムで表示されます...')
    print('-' * 60)

    # リアルタイムに stdout を流す
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1
    )
    for line in proc.stdout:
        print(line, end='')
    proc.wait()

    print('-' * 60)
    if proc.returncode == 0:
        print('\n✅ DLC TopView 解析完了: outputs/result_dlc_topview/')
        print('   ダッシュボードの「比較解析結果」ドロップダウンから結果をロードできます。')
    else:
        print(f'\n⚠️  パイプラインがエラー終了しました (code={proc.returncode})')

In [ ]:
# DLC SuperAnimal-Quadruped で解析する (側面/斜め視点・paws 含む)
import os, sys, subprocess

os.chdir(LOCAL_PROJECT_PATH)

# 実行デバイスは GPU/CPU を自動判別（デバイス選択セル未実行でも安全に動作）
if 'device_args' in globals():
    _device = device_args()
else:
    try:
        import torch
        _device = ['--device', 'cuda' if torch.cuda.is_available() else 'cpu']
    except Exception:
        _device = ['--device', 'cpu']

# ▼ 側面または斜め視点で撮影された動画のパスを指定してください
TARGET_VIDEO_SIDE = '/content/drive/MyDrive/mouse_videos/cage_side.mp4'

if not os.path.exists(TARGET_VIDEO_SIDE):
    print(f'⚠️  動画ファイルが見つかりません: {TARGET_VIDEO_SIDE}')
    print('   TARGET_VIDEO_SIDE を自分の動画パスに変更してから実行してください。')
    print('   例: /content/drive/MyDrive/mouse_videos/cage_side.mp4')
    print()
    print('   ℹ️  Quadruped モデルは paws を直接含むため、ひっかき検出では解剖学的に最適です。')
    print('      ただし側面・斜め視点の動画専用です。真上視点なら cell 上の TopView を使ってください。')
else:
    cmd = [
        sys.executable, 'pipeline.py',
        '--video', TARGET_VIDEO_SIDE,
        '--engine', 'dlc',
        '--dlc-superanimal', 'superanimal_quadruped',
        '--out_dir', 'outputs/result_dlc_quadruped',
        '--px-to-mm', '0.25',
    ] + _device
    print('▶ 実行:', ' '.join(cmd))
    print('   Quadruped モデルは paws を直接含むため、ひっかき検出には解剖学的に最適です。')
    print('-' * 60)

    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1
    )
    for line in proc.stdout:
        print(line, end='')
    proc.wait()

    print('-' * 60)
    if proc.returncode == 0:
        print('\n✅ DLC Quadruped 解析完了: outputs/result_dlc_quadruped/')
    else:
        print(f'\n⚠️  パイプラインがエラー終了しました (code={proc.returncode})')

In [ ]:
# (応用) 全エンジン (YOLOv8 / DLC-TopView / DLC-Quadruped) で
#        同一動画を解析し、ダッシュボードのドロップダウンから精度比較する
import os, sys, subprocess

os.chdir(LOCAL_PROJECT_PATH)

# 実行デバイスは GPU/CPU を自動判別（デバイス選択セル未実行でも安全に動作）
if 'device_args' in globals():
    _device = device_args()
else:
    try:
        import torch
        _device = ['--device', 'cuda' if torch.cuda.is_available() else 'cpu']
    except Exception:
        _device = ['--device', 'cpu']

# ▼ 比較に使う動画パスを指定してください
TARGET_VIDEO = '/content/drive/MyDrive/mouse_videos/cage_01.mp4'

if not os.path.exists(TARGET_VIDEO):
    print(f'⚠️  動画ファイルが見つかりません: {TARGET_VIDEO}')
    print('   TARGET_VIDEO を自分の動画パスに変更してから実行してください。')
else:
    tasks = [
        ('YOLOv8 (人間 COCO 流用・軽量)',
         ['--engine', 'yolov8'],
         'outputs/cmp_yolov8'),
        ('DLC SuperAnimal-TopViewMouse (真上視点・27点・zero-shot)',
         ['--engine', 'dlc', '--dlc-superanimal', 'superanimal_topviewmouse'],
         'outputs/cmp_dlc_topview'),
        ('DLC SuperAnimal-Quadruped (側面視点・39点・paws 含む)',
         ['--engine', 'dlc', '--dlc-superanimal', 'superanimal_quadruped'],
         'outputs/cmp_dlc_quadruped'),
    ]

    results = []
    for i, (name, engine_args, out_dir) in enumerate(tasks, 1):
        print(f'\n{"="*60}')
        print(f'[{i}/{len(tasks)}] {name}')
        print('='*60)
        cmd = [sys.executable, 'pipeline.py', '--video', TARGET_VIDEO,
               '--out_dir', out_dir, '--px-to-mm', '0.25'] + engine_args + _device

        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1
        )
        for line in proc.stdout:
            print(line, end='')
        proc.wait()

        ok = proc.returncode == 0
        results.append((name, out_dir, ok))
        print(f'\n  {"✅" if ok else "⚠️ "} {"完了" if ok else "エラー"}: {out_dir}')

    print('\n' + '='*60)
    print(' 比較解析サマリー')
    print('='*60)
    for name, out_dir, ok in results:
        mark = '✅' if ok else '⚠️ '
        print(f'  {mark} {name}')
        print(f'     → {out_dir}/')
    print()
    print('ダッシュボードの「比較解析結果」ドロップダウンから結果を切り替えて精度を比較してください。')

---
## 🪫 ステップ12 (無料枠向け): 長尺動画の分割解析（途中切断に強い）

無料 Colab（T4 GPU）は、**長尺動画を1回のセッションで解析しきれない**ことがあります（推論速度が遅く、アイドル/接続の上限で切断されるため）。このステップでは動画を**一定長のセグメントに分割**し、1つずつ解析して **Google Drive に逐次保存**します。

### 仕組みと利点
- ✂️ `ffmpeg` で動画を `SEGMENT_MINUTES` 分ごとに切り出し → 各セグメントを個別に解析
- 💾 1セグメント完了ごとに即 Drive へ保存（**切断されても完了分は失われません**）
- 🔁 **自動再開**: 切断後にセルを再実行すると、Drive に完了済みのセグメントはスキップし、続きから再開
- 🎛️ `ONLY_SEGMENTS = [0]` で「冒頭の1区間だけ」など部分実行も可能

### T4 での目安
| 1セグメント長 | 1セグメントの解析時間(目安) | 使い方 |
|--------------|------------------------------|--------|
| 5 分 | 約 20〜40 分 | 安全重視・確実に1区間ずつ完走 |
| 10 分 | 約 40〜80 分 | バランス（推奨） |

> セッションが切れたら、ランタイムを再接続 → ステップ1・2（Drive マウントとプロジェクトコピー）→ このセルを再実行するだけで続きから進みます。

### 結果の見方（重要）
- 各セグメントは独立した結果（`<動画名>_segNNN_*`）として Drive とダッシュボードに出ます。
- **全体の bout 数**は各セグメントの bout 数の合計、**総スクラッチ秒**も合計で求められます。
- `bout 率`(回/分)・`強度`(秒/分) は観察時間で正規化済みなので、全体値は「合計 bout 数 ÷ 全体の総分」で再計算してください。
- ひとつなぎで可視化したい場合は、次の「結合」セルで `*_merged_*` を作成できます（※境界をまたぐ bout は分割される点に注意）。

---


In [ ]:
# ============================================================
#  分割解析（途中切断に強い・Drive へ逐次保存・自動再開）
# ============================================================
# 無料 Colab(T4) は長尺動画を1セッションで解析しきれない（速度・接続上限）ため、
# 動画を一定長のセグメントに分割し、1つずつ解析して Drive に逐次保存します。
# 切断・失敗しても、このセルを再実行すれば「未完了セグメントから自動再開」します。
import os, sys, subprocess, math, shutil

os.chdir(LOCAL_PROJECT_PATH)

# ===== 設定（ここだけ編集してください）=========================
TARGET_VIDEO    = '/content/drive/MyDrive/mouse_videos/1000.mov'   # 解析する動画
SEGMENT_MINUTES = 10            # 1セグメントの長さ(分)。T4 では 5〜10 分を推奨
PX_TO_MM        = 0.25          # ご自身のカメラで実測した値に置き換えてください
ENGINE_ARGS     = ['--engine', 'dlc', '--dlc-superanimal', 'superanimal_topviewmouse']
ONLY_SEGMENTS   = None          # None=全セグメント / 例: [0]=冒頭のみ / [0,1,2]=先頭3つだけ
# =============================================================

DRIVE_OUTPUTS = '/content/drive/MyDrive/mouse-scratch-dashboard/outputs'

# 実行デバイスを自動判定（GPU があれば cuda、無ければ cpu）
try:
    import torch
    _device = ['--device', 'cuda' if torch.cuda.is_available() else 'cpu']
except Exception:
    _device = ['--device', 'cpu']

def _ffprobe_duration(path):
    r = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', path],
        capture_output=True, text=True)
    try:
        return float(r.stdout.strip())
    except Exception:
        return None

if not os.path.exists(TARGET_VIDEO):
    print(f'⚠️  動画が見つかりません: {TARGET_VIDEO}')
    print('   TARGET_VIDEO のパスを修正してから実行してください。')
else:
    dur = _ffprobe_duration(TARGET_VIDEO)
    if dur is None:
        print('⚠️  動画の長さを取得できませんでした（ffprobe 失敗）。動画形式を確認してください。')
    else:
        seg_len = SEGMENT_MINUTES * 60
        n_seg   = math.ceil(dur / seg_len)
        base    = os.path.splitext(os.path.basename(TARGET_VIDEO))[0]
        os.makedirs(DRIVE_OUTPUTS, exist_ok=True)
        seg_tmp = '/content/_segments'
        os.makedirs(seg_tmp, exist_ok=True)

        print(f'🎬 {base}: 全長 {dur/60:.1f} 分 → {SEGMENT_MINUTES} 分 × {n_seg} セグメント')
        print(f'   デバイス: {_device[1]} / エンジン: {" ".join(ENGINE_ARGS)}')
        print('=' * 60)

        targets = range(n_seg) if ONLY_SEGMENTS is None else ONLY_SEGMENTS
        for i in targets:
            seg_name  = f'{base}_seg{i:03d}'
            drive_seg = os.path.join(DRIVE_OUTPUTS, seg_name)
            done_csv  = os.path.join(drive_seg, f'{seg_name}_summary.csv')

            # 既に Drive に完了済みならスキップ（再開対応）
            if os.path.exists(done_csv):
                print(f'⏭️  [{i+1}/{n_seg}] {seg_name}: 完了済み(Drive) → スキップ')
                continue

            start = i * seg_len
            seg_path = os.path.join(seg_tmp, f'{seg_name}.mp4')
            print(f'\n✂️  [{i+1}/{n_seg}] {seg_name}: '
                  f'{start/60:.1f}〜{min(dur, start+seg_len)/60:.1f} 分を切り出し中...')
            # 再エンコード(libx264/yuv420p)で正確な区間切り出し＆OpenCV/DLC 互換に
            cut = subprocess.run(
                ['ffmpeg', '-y', '-ss', str(start), '-i', TARGET_VIDEO, '-t', str(seg_len),
                 '-c:v', 'libx264', '-preset', 'veryfast', '-pix_fmt', 'yuv420p', '-an', seg_path],
                capture_output=True, text=True)
            if not os.path.exists(seg_path) or os.path.getsize(seg_path) == 0:
                print(f'   ⚠️ 切り出し失敗: {cut.stderr[-300:]}')
                break

            out_dir = f'outputs/{seg_name}'
            cmd = [sys.executable, 'pipeline.py', '--video', seg_path, '--out_dir', out_dir,
                   '--px-to-mm', str(PX_TO_MM)] + ENGINE_ARGS + _device
            print('   ▶', ' '.join(cmd))
            print('   ' + '-' * 56)
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                    text=True, encoding='utf-8', errors='replace', bufsize=1)
            for line in proc.stdout:
                print('   ' + line, end='')
            proc.wait()

            if proc.returncode == 0 and os.path.isdir(out_dir):
                # 完了したセグメントの結果を即 Drive に保存（切断対策）
                os.makedirs(drive_seg, exist_ok=True)
                for f in os.listdir(out_dir):
                    if f.endswith(('.csv', '.json')):
                        shutil.copy2(os.path.join(out_dir, f), os.path.join(drive_seg, f))
                try:
                    os.remove(seg_path)  # ローカルディスク節約（切り出し動画を削除）
                except OSError:
                    pass
                print(f'   ✅ {seg_name} 完了 → Drive 保存: {drive_seg}')
            else:
                print(f'   ⚠️ {seg_name} 失敗 (code={proc.returncode})。'
                      f'このセルを再実行すると同セグメントから再開します。')
                break

        print('\n' + '=' * 60)
        print('完了しました。途中で切断/失敗した場合は、このセルを再実行すると')
        print('未完了セグメントから自動的に再開します（完了済みはスキップ）。')
        print('結果をひとつなぎで見たい場合は、次の「結合」セルを実行してください。')

In [ ]:
# ============================================================
#  (任意) 分割結果を1本に結合して、ダッシュボードに通しで読み込む
# ============================================================
# 各セグメントの frame_labels / bouts を、フレーム番号・時刻のオフセットを
# 補正しながら連結し、{動画名}_merged_*.csv を作成します。
#
#  ⚠️ 学術的注意: セグメント境界をまたぐ bout は2つに分割されます。
#     境界は SEGMENT_MINUTES 分おき（通常は数分おき）で、ひっかき bout は
#     秒オーダーのため影響は軽微ですが、厳密性が必要なら SEGMENT_MINUTES を
#     大きくして境界数を減らしてください。各セグメントを個別に見るのが最も正確です。
import os, glob, pandas as pd

os.chdir(LOCAL_PROJECT_PATH)
DRIVE_OUTPUTS = '/content/drive/MyDrive/mouse-scratch-dashboard/outputs'

# 上の分割セルで使った TARGET_VIDEO を流用（未定義なら手動指定してください）
try:
    BASE = os.path.splitext(os.path.basename(TARGET_VIDEO))[0]
except NameError:
    BASE = '1000'  # ← 必要に応じて動画名（拡張子なし）に変更
FPS = 30.0  # フレーム→秒換算。動画の実 fps に合わせてください（例 29.97）

seg_dirs = sorted(glob.glob(os.path.join(DRIVE_OUTPUTS, f'{BASE}_seg*')))
if not seg_dirs:
    print(f'⚠️ 結合対象が見つかりません: {DRIVE_OUTPUTS}/{BASE}_seg*')
    print('   先に「分割解析」セルを実行してください。')
else:
    frame_parts, bout_parts = [], []
    frame_offset = 0
    for d in seg_dirs:
        name = os.path.basename(d)
        fl = os.path.join(d, f'{name}_frame_labels.csv')
        bt = os.path.join(d, f'{name}_bouts.csv')
        if not os.path.exists(fl):
            print(f'  ⏭️ {name}: frame_labels なし → スキップ'); continue
        df = pd.read_csv(fl)
        seg_frames = (int(df['frame_idx'].max()) + 1) if not df.empty else 0
        df['frame_idx'] = df['frame_idx'] + frame_offset
        frame_parts.append(df)
        if os.path.exists(bt):
            bd = pd.read_csv(bt)
            if not bd.empty:
                for c in ('bout_start_frame', 'bout_end_frame'):
                    if c in bd.columns: bd[c] = bd[c] + frame_offset
                for c in ('bout_start_time_s', 'bout_end_time_s'):
                    if c in bd.columns: bd[c] = bd[c] + frame_offset / FPS
                bout_parts.append(bd)
        frame_offset += seg_frames
        print(f'  ✅ {name}: {seg_frames} フレーム連結（累計 {frame_offset}）')

    if frame_parts:
        os.makedirs('outputs', exist_ok=True)
        merged_fl = pd.concat(frame_parts, ignore_index=True)
        out_fl_drive = os.path.join(DRIVE_OUTPUTS, f'{BASE}_merged_frame_labels.csv')
        out_fl_local = os.path.join('outputs', f'{BASE}_merged_frame_labels.csv')
        merged_fl.to_csv(out_fl_drive, index=False)
        merged_fl.to_csv(out_fl_local, index=False)  # ダッシュボード(サーバー)が読むのはローカル outputs/
        print(f'\n💾 結合 frame_labels: {out_fl_local} ({len(merged_fl)} 行)')
        if bout_parts:
            merged_bt = pd.concat(bout_parts, ignore_index=True)
            merged_bt.to_csv(os.path.join(DRIVE_OUTPUTS, f'{BASE}_merged_bouts.csv'), index=False)
            merged_bt.to_csv(os.path.join('outputs', f'{BASE}_merged_bouts.csv'), index=False)
            print(f'💾 結合 bouts: outputs/{BASE}_merged_bouts.csv ({len(merged_bt)} 行)')
        print('\n👉 ダッシュボードで「サーバー結果リストを更新」→ *_merged_* を選ぶと通しで表示できます。')